In [2]:
import pandas as pd
import numpy as np 


import openmeteo_requests
import requests_cache

from retry import retry

import requests
from urllib3.util import Retry
from requests.adapters import HTTPAdapter


In [7]:
#ONLY WEATHER FEATURE

# ========================================================
# SET PATHS AND LOAD ORIGINAL DATASET
# ========================================================
ORG_DISTANCE_INCLUDED_DATASET_PATH = "../data_orig/org_dataset_with_filtered_distance_speed.csv"
DISTANCE_INCLUDED_DATASET_PATH = "../data/dataset_with_filtered_distance_speed.csv"

print("Loading original dataset...")
dataset_df = pd.read_csv(ORG_DISTANCE_INCLUDED_DATASET_PATH)
# ========================================================
# PART 2: FETCH HISTORICAL WEATHER DATA VIA API (STABLE JSON)
# ========================================================
print("Fetching historical weather data from Open-Meteo for London...")

url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": 51.5074,   # London Latitude
    "longitude": -0.1278,  # London Longitude
    "start_date": "2021-01-01",
    "end_date": "2026-02-28",
    "hourly": ["temperature_2m", "rain", "snowfall"],
    "format": "json"       # Explicitly request standard JSON format
}

response = requests.get(url, params=params)

if response.status_code == 200:
    data_json = response.json()
    hourly_data = data_json["hourly"]
    
    # Convert API arrays directly into a clean weather DataFrame
    weather_data = pd.DataFrame({
        "date": pd.to_datetime(hourly_data["time"]),
        "weather_temperature": hourly_data["temperature_2m"],
        "weather_rain": hourly_data["rain"],
        "weather_snowfall": hourly_data["snowfall"]
    })
    weather_data['date'] = weather_data['date'].dt.tz_localize(None)
    
    # Extract keys for mapping with dataset_df
    weather_data['CalYear'] = weather_data['date'].dt.year
    weather_data['Month'] = weather_data['date'].dt.month
    weather_data['Weekday'] = weather_data['date'].dt.weekday  # 0=Monday, 6=Sunday
    weather_data['Hour'] = weather_data['date'].dt.hour
    print("Weather data successfully received in JSON format.")
else:
    raise RuntimeError(f"API Error: Status-Code {response.status_code}. Details: {response.text}")


# ========================================================
# PART 3: AGGREGATE WEATHER & LOCAL MERGE
# ========================================================
print("Aggregating weather data by time categories...")
weather_agg = (
    weather_data.groupby(['CalYear', 'Month', 'Weekday', 'Hour'])
    .agg({
        'weather_temperature': 'mean',
        'weather_rain': 'mean',
        'weather_snowfall': 'mean',
    })
    .reset_index()
)

# Merge weather with the main dataset (executed locally via Pandas)
dataset_df = pd.merge(
    dataset_df, weather_agg, on=['CalYear', 'Month', 'Weekday', 'Hour'], how='left'
)


# ========================================================
# PART 4: BIN WEATHER DATA INTO CATEGORICAL BUCKETS
# ========================================================
print("Binning weather data into categorical buckets...")

# 1. Temperature Buckets
temp_bins = [-99, 0, 10, 22, 99]
temp_labels = ['temp_freezing', 'temp_cold', 'temp_mild', 'temp_hot']
dataset_df['weather_temp_bucket'] = pd.cut(
    dataset_df['weather_temperature'], bins=temp_bins, labels=temp_labels
).astype(str)

# 2. Rain Buckets
rain_bins = [-1, 0.01, 0.5, 2.5, 99]
rain_labels = ['rain_none', 'rain_light', 'rain_moderate', 'rain_heavy']
dataset_df['weather_rain_bucket'] = pd.cut(
    dataset_df['weather_rain'], bins=rain_bins, labels=rain_labels
).astype(str)

# 3. Snow Buckets
snow_bins = [-1, 0.01, 99]
snow_labels = ['snow_none', 'snow_active']
dataset_df['weather_snow_bucket'] = pd.cut(
    dataset_df['weather_snowfall'], bins=snow_bins, labels=snow_labels
).astype(str)

# 4. Interaction Bucket: Rain during Rush Hour
is_heavy_rain = dataset_df['weather_rain'] > 0.5
is_rush_hour = dataset_df['Is_Rush_Hour'] == 1

dataset_df['rain_rush_hour_bucket'] = 'rush_wetter_normal'
dataset_df.loc[is_rush_hour & ~is_heavy_rain, 'rain_rush_hour_bucket'] = 'rush_only'
dataset_df.loc[~is_rush_hour & is_heavy_rain, 'rain_rush_hour_bucket'] = 'rain_only'
dataset_df.loc[is_rush_hour & is_heavy_rain, 'rain_rush_hour_bucket'] = 'rush_and_heavy_rain'

# 5. Interaction Bucket: Road Ice Risk (Cold AND Snowing)
is_freezing = dataset_df['weather_temperature'] <= 2
has_snow = dataset_df['weather_snowfall'] > 0

dataset_df['road_ice_bucket'] = 'ice_risk_low'
dataset_df.loc[is_freezing & has_snow, 'road_ice_bucket'] = 'ice_risk_extreme'

# Remove raw numeric API weather columns to prevent redundancy
dataset_df = dataset_df.drop(
    columns=['weather_temperature', 'weather_rain', 'weather_snowfall']
)


# ========================================================
# PART 5: FINAL SAVE (ONCE AT THE END)
# ========================================================
print("Saving the final extended dataset...")
dataset_df.to_csv(DISTANCE_INCLUDED_DATASET_PATH, index=False)
print("Successfully finished! All 5 weather buckets are saved in the CSV.")

Loading original dataset...
Fetching historical weather data from Open-Meteo for London...
Weather data successfully received in JSON format.
Aggregating weather data by time categories...
Binning weather data into categorical buckets...
Saving the final extended dataset...
Successfully finished! All 5 weather buckets are saved in the CSV.


In [ ]:
# New Idea for rolling windows

# ========================================================
# SET PATHS AND LOAD ORIGINAL DATASET
# ========================================================
ORG_DISTANCE_INCLUDED_DATASET_PATH = "../data_orig/org_dataset_with_filtered_distance_speed.csv"
DISTANCE_INCLUDED_DATASET_PATH = "../data/dataset_with_filtered_distance_speed.csv"

print("Loading original dataset...")
dataset_df = pd.read_csv(ORG_DISTANCE_INCLUDED_DATASET_PATH)

# ========================================================
# PART 1: COMPUTE ROLLING WINDOWS (ROW-BASED FOR LACK OF DATE)
# ========================================================
print("Computing row-based rolling windows (including speed/inertia)...")
dataset_df = dataset_df.reset_index(drop=True)

# --- calulating speed ---
dataset_df['_current_inertia'] = dataset_df['distance_fire_to_station'] / dataset_df['AttendanceTimeSeconds']


# --- FEATURE 1: ROLLING MEAN PER FIRE STATION ---
dataset_df = dataset_df.sort_values(by=['DeployedFromStation_Name', 'CalYear', 'Month', 'Hour'])

dataset_df['_prev_attr_station'] = dataset_df.groupby('DeployedFromStation_Name')['AttendanceTimeSeconds'].shift(1)
dataset_df['_prev_inertia_station'] = dataset_df.groupby('DeployedFromStation_Name')['_current_inertia'].shift(1)

rolling_station = dataset_df.groupby('DeployedFromStation_Name')[['_prev_attr_station', '_prev_inertia_station']].rolling(window=3, min_periods=1)
rolling_station_df = rolling_station.mean().reset_index(level=0, drop=True).sort_index()

# important!
dataset_df = dataset_df.sort_index()

dataset_df['station_rolling_mean_last_3'] = rolling_station_df['_prev_attr_station']
dataset_df['station_inertia_rolling_mean_last_3'] = rolling_station_df['_prev_inertia_station']

# Fallbacks
backup_station_time = dataset_df.groupby('DeployedFromStation_Name')['AttendanceTimeSeconds'].transform('mean')
backup_station_inertia = dataset_df.groupby('DeployedFromStation_Name')['_current_inertia'].transform('mean')

dataset_df['station_rolling_mean_last_3'] = dataset_df['station_rolling_mean_last_3'].fillna(backup_station_time)
dataset_df['station_inertia_rolling_mean_last_3'] = dataset_df['station_inertia_rolling_mean_last_3'].fillna(backup_station_inertia)


# --- FEATURE 2: ROLLING MEAN & MAX PER BOROUGH ---
dataset_df = dataset_df.sort_values(by=['IncGeo_BoroughName', 'CalYear', 'Month', 'Hour'])

dataset_df['_prev_attr_borough'] = dataset_df.groupby('IncGeo_BoroughName')['AttendanceTimeSeconds'].shift(1)
dataset_df['_prev_inertia_borough'] = dataset_df.groupby('IncGeo_BoroughName')['_current_inertia'].shift(1)

rolling_borough = dataset_df.groupby('IncGeo_BoroughName')[['_prev_attr_borough', '_prev_inertia_borough']].rolling(window=3, min_periods=1)

# for mean with sorted index
rolling_borough_mean = rolling_borough.mean().reset_index(level=0, drop=True).sort_index()

# for max with sorted index
rolling_borough_max = rolling_borough.max().reset_index(level=0, drop=True).sort_index()

# important!
dataset_df = dataset_df.sort_index()

dataset_df['borough_rolling_mean_last_3'] = rolling_borough_mean['_prev_attr_borough']
dataset_df['borough_inertia_rolling_mean_last_3'] = rolling_borough_mean['_prev_inertia_borough']
dataset_df['borough_rolling_max_last_3'] = rolling_borough_max['_prev_attr_borough']
dataset_df['borough_inertia_rolling_max_last_3'] = rolling_borough_max['_prev_inertia_borough']

# Fallbacks
backup_borough_time = dataset_df.groupby('IncGeo_BoroughName')['AttendanceTimeSeconds'].transform('mean')
backup_borough_inertia = dataset_df.groupby('IncGeo_BoroughName')['_current_inertia'].transform('mean')

dataset_df['borough_rolling_mean_last_3'] = dataset_df['borough_rolling_mean_last_3'].fillna(backup_borough_time)
dataset_df['borough_rolling_max_last_3'] = dataset_df['borough_rolling_max_last_3'].fillna(backup_borough_time)
dataset_df['borough_inertia_rolling_mean_last_3'] = dataset_df['borough_inertia_rolling_mean_last_3'].fillna(backup_borough_inertia)
dataset_df['borough_inertia_rolling_max_last_3'] = dataset_df['borough_inertia_rolling_max_last_3'].fillna(backup_borough_inertia)


# --- more added features
dataset_df['expected_time_by_station_inertia'] = (
    dataset_df['distance_fire_to_station'] / (dataset_df['station_inertia_rolling_mean_last_3'] + 0.1)
)

dataset_df['expected_time_by_borough_inertia'] = (
    dataset_df['distance_fire_to_station'] / (dataset_df['borough_inertia_rolling_mean_last_3'] + 0.1)
)

# --- FINAL CLEANUP & RESORT ---
dataset_df = dataset_df.drop(columns=[
    '_current_inertia', 
    '_prev_attr_station', '_prev_inertia_station', 
    '_prev_attr_borough', '_prev_inertia_borough'
])

dataset_df = dataset_df.sort_index()
print("Part 1 erfolgreich und mathematisch fehlerfrei beendet!")

#ONLY WEATHER FEATURE


# ========================================================
# PART 2: FETCH HISTORICAL WEATHER DATA VIA API (STABLE JSON)
# ========================================================
print("Fetching historical weather data from Open-Meteo for London...")

url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": 51.5074,   # London Latitude
    "longitude": -0.1278,  # London Longitude
    "start_date": "2021-01-01",
    "end_date": "2026-02-28",
    "hourly": ["temperature_2m", "rain", "snowfall"],
    "format": "json"       # Explicitly request standard JSON format
}

response = requests.get(url, params=params)

if response.status_code == 200:
    data_json = response.json()
    hourly_data = data_json["hourly"]
    
    # Convert API arrays directly into a clean weather DataFrame
    weather_data = pd.DataFrame({
        "date": pd.to_datetime(hourly_data["time"]),
        "weather_temperature": hourly_data["temperature_2m"],
        "weather_rain": hourly_data["rain"],
        "weather_snowfall": hourly_data["snowfall"]
    })
    weather_data['date'] = weather_data['date'].dt.tz_localize(None)
    
    # Extract keys for mapping with dataset_df
    weather_data['CalYear'] = weather_data['date'].dt.year
    weather_data['Month'] = weather_data['date'].dt.month
    weather_data['Weekday'] = weather_data['date'].dt.weekday  # 0=Monday, 6=Sunday
    weather_data['Hour'] = weather_data['date'].dt.hour
    print("Weather data successfully received in JSON format.")
else:
    raise RuntimeError(f"API Error: Status-Code {response.status_code}. Details: {response.text}")


# ========================================================
# PART 3: AGGREGATE WEATHER & LOCAL MERGE
# ========================================================
print("Aggregating weather data by time categories...")
weather_agg = (
    weather_data.groupby(['CalYear', 'Month', 'Weekday', 'Hour'])
    .agg({
        'weather_temperature': 'mean',
        'weather_rain': 'mean',
        'weather_snowfall': 'mean',
    })
    .reset_index()
)

# Merge weather with the main dataset (executed locally via Pandas)
dataset_df = pd.merge(
    dataset_df, weather_agg, on=['CalYear', 'Month', 'Weekday', 'Hour'], how='left'
)


# ========================================================
# PART 4: BIN WEATHER DATA INTO CATEGORICAL BUCKETS
# ========================================================
print("Binning weather data into categorical buckets...")

# 1. Temperature Buckets
temp_bins = [-99, 0, 10, 22, 99]
temp_labels = ['temp_freezing', 'temp_cold', 'temp_mild', 'temp_hot']
dataset_df['weather_temp_bucket'] = pd.cut(
    dataset_df['weather_temperature'], bins=temp_bins, labels=temp_labels
).astype(str)

# 2. Rain Buckets
rain_bins = [-1, 0.01, 0.5, 2.5, 99]
rain_labels = ['rain_none', 'rain_light', 'rain_moderate', 'rain_heavy']
dataset_df['weather_rain_bucket'] = pd.cut(
    dataset_df['weather_rain'], bins=rain_bins, labels=rain_labels
).astype(str)

# 3. Snow Buckets
snow_bins = [-1, 0.01, 99]
snow_labels = ['snow_none', 'snow_active']
dataset_df['weather_snow_bucket'] = pd.cut(
    dataset_df['weather_snowfall'], bins=snow_bins, labels=snow_labels
).astype(str)

# 4. Interaction Bucket: Rain during Rush Hour
is_heavy_rain = dataset_df['weather_rain'] > 0.5
is_rush_hour = dataset_df['Is_Rush_Hour'] == 1

dataset_df['rain_rush_hour_bucket'] = 'rush_wetter_normal'
dataset_df.loc[is_rush_hour & ~is_heavy_rain, 'rain_rush_hour_bucket'] = 'rush_only'
dataset_df.loc[~is_rush_hour & is_heavy_rain, 'rain_rush_hour_bucket'] = 'rain_only'
dataset_df.loc[is_rush_hour & is_heavy_rain, 'rain_rush_hour_bucket'] = 'rush_and_heavy_rain'

# 5. Interaction Bucket: Road Ice Risk (Cold AND Snowing)
is_freezing = dataset_df['weather_temperature'] <= 2
has_snow = dataset_df['weather_snowfall'] > 0

dataset_df['road_ice_bucket'] = 'ice_risk_low'
dataset_df.loc[is_freezing & has_snow, 'road_ice_bucket'] = 'ice_risk_extreme'

# Remove raw numeric API weather columns to prevent redundancy
dataset_df = dataset_df.drop(
    columns=['weather_temperature', 'weather_rain', 'weather_snowfall']
)


# ========================================================
# PART 5: FINAL SAVE (ONCE AT THE END)
# ========================================================
print("Saving the final extended dataset...")
dataset_df.to_csv(DISTANCE_INCLUDED_DATASET_PATH, index=False)
print("Successfully finished! All 8 rolling windows and 5 weather buckets are saved in the CSV.")

Loading original dataset...
Computing row-based rolling windows (including speed/inertia)...
Part 1 erfolgreich und mathematisch fehlerfrei beendet!
Fetching historical weather data from Open-Meteo for London...
Weather data successfully received in JSON format.
Aggregating weather data by time categories...
Binning weather data into categorical buckets...
Saving the final extended dataset...
Successfully finished! All 5 rolling windows and 5 weather buckets are saved in the CSV.


In [12]:
dataset_df.head()




,index,IncidentNumber,CalYear,Month,Weekday,Hour,Is_Nightshift,Is_Rush_Hour,Is_Weekend,Is_Public_Holiday,...,borough_inertia_rolling_mean_last_3,borough_rolling_max_last_3,borough_inertia_rolling_max_last_3,expected_time_by_station_inertia,expected_time_by_borough_inertia,weather_temp_bucket,weather_rain_bucket,weather_snow_bucket,rain_rush_hour_bucket,road_ice_bucket
0,0,000004-01012021,2021,1,4,0,1,0,0,1,...,6.960226,322.242463,6.960226,205.152131,211.083906,temp_cold,rain_moderate,snow_none,rain_only,ice_risk_low
1,1,000005-01012021,2021,1,4,0,1,0,0,1,...,8.378528,347.711628,8.378528,141.987658,137.571052,temp_cold,rain_moderate,snow_none,rain_only,ice_risk_low
2,2,000006-01012021,2021,1,4,0,1,0,0,1,...,8.482140,342.634607,8.482140,393.539178,403.582318,temp_cold,rain_moderate,snow_none,rain_only,ice_risk_low
3,3,000007-01012021,2021,1,4,0,1,0,0,1,...,6.951032,304.507660,6.951032,246.081002,280.185940,temp_cold,rain_moderate,snow_none,rain_only,ice_risk_low
4,4,000009-01012021,2021,1,4,0,1,0,0,1,...,7.223745,316.497264,7.223745,151.959466,143.287345,temp_cold,rain_moderate,snow_none,rain_only,ice_risk_low
